# 监督微调：模型训练

上一章，我们构建了 SFT 训练数据：512 个（提示，回答）数据对。本章，我们将用这些数据对预训练模型进行**监督微调**。

## 监督微调

监督微调的核心思想：**只计算回答部分的损失**。

预训练时，模型对每个位置都计算损失，因为每个位置都在学习下一个词元。但到了监督微调阶段，我们只关心模型是否学会了**生成合适的答案**，并不想用提示部分影响模型。

### 掩码

解决办法就是给每个样本加一个**掩码**（Mask）：

* 提示部分：掩码为 0，不计算损失；
* 回答部分：掩码为 1，正常计算损失。

这样，模型只会从回答部分学习，不会被提示部分干扰。

### 策略

* **学习率**：监督微调通常采用比预训练阶段小一个数量级的学习率。因为模型已经训练好了，我们只想微调模型生成文本的方向，而不是知识逻辑；
* **轮数**：通常只训练 1 个轮次。SFT 数据量小，训练太久容易过拟合；

In [1]:
import pickle

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

## 数据集

### SFT 数据集

**SFT 数据集** SFTDataset 和预训练的数据集不同，想要处理（提示，回答）数据对：

* **拼接**：把提示和回答拼接到一起，成为一个词元序列；
* **截断**：如果超出上下文尺寸，从提示部分的头部截断，保留完整的回答部分；
* **掩码**：给提示部分添加掩码 0，给回答部分添加掩码 1。结果返回 `(特征值, 标签值, 掩码)`。

In [3]:
class SFTDataset(Dataset):

    def __init__(self, filename, context_size=64, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.split = split
        super().__init__(1)

    def load(self):
        with open(self.filename, "rb") as f:
            examples = pickle.load(f)

        split = int(len(examples) * self.split)
        self.train_data = self._pack(examples[:split])
        self.test_data = self._pack(examples[split:])

    def _pack(self, examples):
        xs, ys, masks = [], [], []
        for prompt, response in examples:
            x, y, mask = self._build_example(prompt, response)
            xs.append(x)
            ys.append(y)
            masks.append(mask)
        return xs, ys, masks

    def _build_example(self, prompt, response):
        tokens = list(prompt) + list(response)
        if len(tokens) > self.context_size + 1:
            overflow = len(tokens) - (self.context_size + 1)
            prompt = prompt[overflow:] if overflow < len(prompt) else []
            tokens = list(prompt) + list(response)
            tokens = tokens[-(self.context_size + 1):]

        response_start = len(prompt)

        x = np.array(tokens[:-1], dtype=np.int64)
        y = np.array(tokens[1:], dtype=np.int64)
        mask = np.arange(len(y)) + 1 >= response_start
        return x, y, mask

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y, mask = self.data
        return Tensor(x[s]), Tensor(y[s]), Tensor(mask[s])

## 损失函数

### SFT 交叉熵损失函数

**SFT 损失函数** SFTLoss 是带掩码的交叉熵损失函数：

* 计算所有位置的交叉熵；
* 乘以掩码，只保留答案部分的损失；
* 除以答案部分的长度（掩码中 1 的个数），求取有效值的平均。

反向传播时，梯度也根据掩码，只通过答案部分传回。不影响提示部分。

In [4]:
class SFTLoss(Loss):

    def loss(self, p: Tensor, y: Tensor, mask):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        flat_softmax = softmax.reshape(-1, softmax.shape[-1])
        flat_y = y.data.reshape(-1).astype(np.int64)
        flat_mask = mask.data.reshape(-1)

        rows = np.arange(len(flat_y))
        n = max(np.sum(flat_mask), 1.0)

        log = np.log(np.clip(flat_softmax[rows, flat_y], 1e-10, 1))
        ce = Tensor(0 - np.sum(log * flat_mask) / n)

        def gradient_fn():
            flat_grad = flat_softmax.copy()
            flat_grad[rows, flat_y] -= 1
            flat_grad *= flat_mask[:, None]
            p.grad += ce.grad * flat_grad.reshape(softmax.shape) / n

        return ce.attach(gradient_fn, parents={p})

## 训练器

### SFT 训练器

**SFT 训练器** SFTTrainer 继承自 GPT 训练器 GTPTrainer。有一下主要区别：

* **train**（训练函数）：从数据集接收掩码，作为参数传给损失函数；
* **test**（测试函数）：监督微调阶段不需要测试，直接返回 None。

SFT 只做 1 轮训练，结束后直接保存模型参数。

In [5]:
class SFTTrainer(GPTTrainer):

    def train(self, dataset, epochs, scheduler=None, filename=None):
        self.model.train()

        steps = 0
        for epoch in range(epochs):
            order = list(range(len(dataset)))
            np.random.shuffle(order)

            total_loss = 0.0
            for step, i in enumerate(order):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                feature, label, mask = dataset[i]
                prediction = self.model(feature)
                loss = self.loss_fn(prediction, label, mask)

                self.optimizer.zero_grad()
                loss.backward()
                total_loss += float(loss.data)
                self.optimizer.clip_grad_norm()
                self.optimizer.step()
                steps += 1

                if (step + 1) % 100 == 0:
                    lr = f" lr {self.optimizer.lr:.6f}" if scheduler is not None else ""
                    print(f"epoch {epoch + 1} step {step + 1}/{len(dataset)} loss {(total_loss / 100):.4f}{lr}")
                    total_loss = 0.0

            if filename is not None:
                self.save(filename)
                print(f"epoch {epoch + 1} saved SFT model to {filename}")

    def test(self, dataset):
        return None

## 超参数

### 学习率

监督微调阶段采用更小的学习率，避免过度影响预训练的结果。

In [6]:
LEARNING_RATE = 0.0001

### 模型参数

其他模型参数需要保持和预训练阶段一致。

In [7]:
BATCH_SIZE = 16
CONTEXT_SIZE = 256
EMBEDDING_SIZE = 384
HEADS = 6
BLOCKS = 6

## 建模

### 加载模型

In [8]:
DATA_FILE = "../../../../data/tinyshakespeare.txt"
MODEL_FILE = "../../../../data/tinyshakespeare-gpt.npz"

dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
model = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = SFTLoss()
optimizer = AdamWOptimizer(model.parameters, lr=LEARNING_RATE)
trainer = SFTTrainer(model, loss_fn, optimizer)
trainer.load(MODEL_FILE)

### 加载 SFT 数据

In [9]:
SFT_SAMPLES = "../../../../data/sft-samples.pkl"

sft_dataset = SFTDataset(SFT_SAMPLES, CONTEXT_SIZE)

## 训练

In [10]:
SFT_MODEL = "../../../../data/tinyshakespeare-sft.npz"

scheduler = WarmupCosineScheduler(LEARNING_RATE, len(sft_dataset), 50, LEARNING_RATE / 10)
trainer.train(sft_dataset, 1, scheduler, SFT_MODEL)

epoch 1 step 100/460 loss 1.7120 lr 0.000097
epoch 1 step 200/460 loss 1.4982 lr 0.000074
epoch 1 step 300/460 loss 1.3893 lr 0.000040
epoch 1 step 400/460 loss 1.4020 lr 0.000015
epoch 1 saved SFT model to ../../../../data/tinyshakespeare-sft.npz


## 推理

In [11]:
print(trainer.generate(dataset, prompt="ROMEO:"))

ROMEO:
Such you three appeal.
PETRUCHIO:
He is is not so right quickil cannot, how my every feewel.
HORTENSIO:
Poor bid then eye to unwish palace.
GRUMIO:
Had not taking on-heavens my windly side to my mistress from Right, Lancaster, I hearn go thee story father


本章，我们用 SFT 数据集对预训练模型进行了**监督微调**：只计算回答部分的损失，让模型学会按照指令对话。

但是 SFT 也有自己的局限：它只告诉模型怎样回答问题，但是不能告诉模型什么是**更好的答案**。比如两个答案都正确，但是一个简介明了、另一个啰哩啰嗦。SFT 无法告诉模型那个更好。

下一章，我们将引入**直接偏好优化**（Direct Preferecne Optimization, DPO），让模型学会从多个答案中选出更好的一个。

## 课后练习

试试把 SFT 的学习率提高，比如和预训练一样，观察训练效果如何？思考为什么微调需要更小的学习率？